In [2]:
from pathlib import Path
from collections import defaultdict
import csv
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "/Users/gupta/Documents/DIS-IND/data/uniprot"

CHUNK_SIZE = 200_000


# ============================================================
# NORMALIZE VALUE
# ============================================================

def normalize_value(value):
    """
    Normalize values before distinct-value and cluster analysis.
    """

    if value is None:
        return ""

    value = str(value)

    value = value.replace("\r", " ")
    value = value.replace("\n", " ")

    # IMPORTANT:
    # Do NOT replace tabs here because TSV files use tabs
    # as separators. Once pandas has parsed the file,
    # tab separators are already removed anyway.

    value = " ".join(value.split())

    return value.strip()


# ============================================================
# ANALYZER
# ============================================================

def analyze_dataset(dataset_path, chunk_size=100_000):

    dataset_path = Path(dataset_path)


    # ========================================================
    # FIND FILES
    # ========================================================

    csv_files = sorted(dataset_path.glob("*.csv"))
    tbl_files = sorted(dataset_path.glob("*.tbl"))
    tsv_files = sorted(dataset_path.glob("*.tsv"))


    # --------------------------------------------------------
    # Make sure only ONE type exists in a dataset
    # --------------------------------------------------------

    available_types = []

    if csv_files:
        available_types.append("csv")

    if tbl_files:
        available_types.append("tbl")

    if tsv_files:
        available_types.append("tsv")


    if len(available_types) > 1:

        raise ValueError(
            "Dataset contains multiple file formats: "
            + ", ".join(available_types)
            + ". Use only one format per dataset."
        )


    # --------------------------------------------------------
    # Select files and separator
    # --------------------------------------------------------

    if csv_files:

        files = csv_files
        file_type = "csv"

        # CSV usually comma-separated
        separator = ","


    elif tbl_files:

        files = tbl_files
        file_type = "tbl"

        # Typical TBL / TPC-H separator
        separator = "|"


    elif tsv_files:

        files = tsv_files
        file_type = "tsv"

        # TSV = TAB separated
        separator = "\t"


    else:

        raise FileNotFoundError(
            f"No .csv, .tbl, or .tsv files found in:\n"
            f"{dataset_path}"
        )


    print("=" * 80)
    print("DATASET ANALYSIS")
    print("=" * 80)

    print(f"Dataset path : {dataset_path}")
    print(f"File type    : .{file_type}")
    print(f"Separator    : {repr(separator)}")
    print(f"Tables       : {len(files):,}")


    # ========================================================
    # GLOBAL STATISTICS
    # ========================================================

    total_rows = 0

    max_rows_per_table = 0

    total_attributes = 0

    all_attribute_distinct_counts = []


    # --------------------------------------------------------
    # All distinct values across entire dataset
    # --------------------------------------------------------

    all_dataset_values = set()


    # --------------------------------------------------------
    # For cluster calculation
    #
    # value -> attributes containing value
    #
    # Example:
    #
    # 1 -> {A, B, C}
    # 2 -> {A, B}
    # 3 -> {A, B, D}
    # 4 -> {A, B}
    #
    # Gives 3 unique clusters.
    # --------------------------------------------------------

    value_to_attributes = defaultdict(set)


    # --------------------------------------------------------
    # Store table-level report
    # --------------------------------------------------------

    table_reports = []


    # ========================================================
    # TOTAL DATASET SIZE
    # ========================================================

    total_size_bytes = sum(
        file.stat().st_size
        for file in files
    )

    total_size_mb = (
        total_size_bytes
        / (1024 * 1024)
    )


    # ========================================================
    # PROCESS EACH TABLE
    # ========================================================

    for file_path in files:

        print()
        print("=" * 80)
        print(f"Analyzing: {file_path.name}")
        print("=" * 80)


        table_rows = 0

        table_size_mb = (
            file_path.stat().st_size
            / (1024 * 1024)
        )


        # ====================================================
        # GET COLUMN NAMES
        # ====================================================

        # ----------------------------------------------------
        # CSV and TSV normally contain headers
        # ----------------------------------------------------

        if file_type in ("csv", "tsv"):

            header = pd.read_csv(
                file_path,

                sep=separator,

                nrows=0,

                engine="python",

                quoting=csv.QUOTE_NONE,

                skip_blank_lines=True
            )

            columns = [
                str(column).strip()
                for column in header.columns
            ]


        # ----------------------------------------------------
        # TBL usually has NO header
        # ----------------------------------------------------

        else:

            sample = pd.read_csv(
                file_path,

                sep=separator,

                header=None,

                nrows=1,

                dtype=str,

                engine="python",

                quoting=csv.QUOTE_NONE,

                skip_blank_lines=True
            )


            # ------------------------------------------------
            # TBL frequently ends with |
            #
            # Example:
            #
            # 1|ABC|100|
            #
            # This creates an empty final column.
            # ------------------------------------------------

            if (
                len(sample.columns) > 0
                and
                sample.iloc[:, -1].isna().all()
            ):

                number_of_columns = (
                    len(sample.columns) - 1
                )

            else:

                number_of_columns = len(
                    sample.columns
                )


            columns = [
                f"column_{i + 1}"
                for i in range(number_of_columns)
            ]


        number_of_attributes = len(columns)

        total_attributes += number_of_attributes


        print(
            f"Detected attributes: "
            f"{number_of_attributes:,}"
        )


        # ====================================================
        # DISTINCT VALUES PER ATTRIBUTE
        # ====================================================

        distinct_values = {

            column: set()

            for column in columns
        }


        # ====================================================
        # CREATE CHUNK READER
        # ====================================================

        # ----------------------------------------------------
        # CSV / TSV WITH HEADER
        # ----------------------------------------------------

        if file_type in ("csv", "tsv"):

            reader = pd.read_csv(
                file_path,

                sep=separator,

                chunksize=chunk_size,

                dtype=str,

                keep_default_na=False,

                engine="python",

                quoting=csv.QUOTE_NONE,

                skip_blank_lines=True,

                on_bad_lines="warn"
            )


        # ----------------------------------------------------
        # TBL WITHOUT HEADER
        # ----------------------------------------------------

        else:

            reader = pd.read_csv(
                file_path,

                sep=separator,

                header=None,

                chunksize=chunk_size,

                dtype=str,

                keep_default_na=False,

                engine="python",

                quoting=csv.QUOTE_NONE,

                skip_blank_lines=True,

                on_bad_lines="warn"
            )


        # ====================================================
        # PROCESS CHUNKS
        # ====================================================

        for chunk in reader:


            # ------------------------------------------------
            # Handle extra trailing column in .tbl
            # ------------------------------------------------

            if (
                file_type == "tbl"
                and
                len(chunk.columns) > len(columns)
            ):

                chunk = chunk.iloc[
                    :,
                    :len(columns)
                ]


            # ------------------------------------------------
            # Safety: too many columns
            # ------------------------------------------------

            if len(chunk.columns) > len(columns):

                chunk = chunk.iloc[
                    :,
                    :len(columns)
                ]


            # ------------------------------------------------
            # Safety: missing columns
            # ------------------------------------------------

            if len(chunk.columns) < len(columns):

                while (
                    len(chunk.columns)
                    < len(columns)
                ):

                    chunk[
                        f"_missing_{len(chunk.columns)}"
                    ] = ""


            chunk.columns = columns


            # =================================================
            # REMOVE COMPLETELY EMPTY ROWS
            # =================================================

            non_empty_mask = (
                chunk
                .astype(str)
                .apply(
                    lambda row: any(
                        normalize_value(value) != ""
                        for value in row
                    ),
                    axis=1
                )
            )


            chunk = chunk[
                non_empty_mask
            ]


            # =================================================
            # COUNT ROWS
            # =================================================

            table_rows += len(chunk)


            # =================================================
            # PROCESS ATTRIBUTES
            # =================================================

            for column in columns:


                # ---------------------------------------------
                # Normalize
                # ---------------------------------------------

                values = chunk[column].map(
                    normalize_value
                )


                # ---------------------------------------------
                # Ignore empty values
                # ---------------------------------------------

                values = values[
                    values != ""
                ]


                # ---------------------------------------------
                # Unique values in this chunk
                # ---------------------------------------------

                unique_values = values.unique()


                # ---------------------------------------------
                # Distinct values for this attribute
                # ---------------------------------------------

                distinct_values[
                    column
                ].update(
                    unique_values
                )


                # ---------------------------------------------
                # Distinct values across whole dataset
                # ---------------------------------------------

                all_dataset_values.update(
                    unique_values
                )


                # ---------------------------------------------
                # Qualified attribute name
                #
                # Example:
                #
                # proteins.Entry
                # proteins.ProteinName
                #
                # or:
                #
                # orders.customer_id
                # customers.customer_id
                # ---------------------------------------------

                qualified_attribute = (
                    f"{file_path.stem}.{column}"
                )


                # ---------------------------------------------
                # Cluster information
                # ---------------------------------------------

                for value in unique_values:

                    value_to_attributes[
                        value
                    ].add(
                        qualified_attribute
                    )


        # ====================================================
        # TABLE STATISTICS
        # ====================================================

        total_rows += table_rows


        max_rows_per_table = max(
            max_rows_per_table,
            table_rows
        )


        table_distinct_counts = []


        for column in columns:

            distinct_count = len(
                distinct_values[column]
            )


            table_distinct_counts.append(
                distinct_count
            )


            all_attribute_distinct_counts.append(
                distinct_count
            )


        # ====================================================
        # TABLE MAX / AVG DISTINCT
        # ====================================================

        if table_distinct_counts:

            table_max_distinct = max(
                table_distinct_counts
            )


            table_avg_distinct = (
                sum(table_distinct_counts)
                /
                len(table_distinct_counts)
            )

        else:

            table_max_distinct = 0

            table_avg_distinct = 0


        # ====================================================
        # SAVE TABLE REPORT
        # ====================================================

        table_reports.append({

            "table":
                file_path.name,

            "size_mb":
                table_size_mb,

            "rows":
                table_rows,

            "attributes":
                number_of_attributes,

            "max_distinct":
                table_max_distinct,

            "average_distinct":
                table_avg_distinct
        })


        # ====================================================
        # PRINT TABLE SUMMARY
        # ====================================================

        print(
            f"Rows               : "
            f"{table_rows:,}"
        )

        print(
            f"Attributes         : "
            f"{number_of_attributes:,}"
        )

        print(
            f"Size (MB)          : "
            f"{table_size_mb:,.2f}"
        )

        print(
            f"Max distinct       : "
            f"{table_max_distinct:,}"
        )

        print(
            f"Average distinct   : "
            f"{table_avg_distinct:,.2f}"
        )


        # Can release table-specific sets now
        del distinct_values


    # ========================================================
    # DATASET-WIDE DISTINCT STATISTICS
    # ========================================================

    if all_attribute_distinct_counts:

        max_distinct_values_per_attribute = max(
            all_attribute_distinct_counts
        )


        average_distinct_values_per_attribute = (
            sum(all_attribute_distinct_counts)
            /
            len(all_attribute_distinct_counts)
        )

    else:

        max_distinct_values_per_attribute = 0

        average_distinct_values_per_attribute = 0


    # ========================================================
    # TOTAL DISTINCT VALUES IN COMPLETE DATASET
    # ========================================================

    total_distinct_values_dataset = len(
        all_dataset_values
    )


    # ========================================================
    # COUNT CLUSTERS ONLY
    # ========================================================
    #
    # Does NOT print individual clusters.
    # ========================================================

    unique_attribute_sets = {

        frozenset(attributes)

        for attributes
        in value_to_attributes.values()
    }


    number_of_clusters = len(
        unique_attribute_sets
    )


    # ========================================================
    # FINAL DATASET SUMMARY
    # ========================================================

    print()
    print()
    print("=" * 90)
    print("FINAL DATASET SUMMARY")
    print("=" * 90)


    print(
        f"File type                              : "
        f".{file_type}"
    )


    print(
        f"Number of tables                       : "
        f"{len(files):,}"
    )


    print(
        f"Dataset size (MB)                      : "
        f"{total_size_mb:,.2f}"
    )


    print(
        f"Total rows                             : "
        f"{total_rows:,}"
    )


    print(
        f"Max # rows / tuples per table          : "
        f"{max_rows_per_table:,}"
    )


    print(
        f"Total # attributes                     : "
        f"{total_attributes:,}"
    )


    print(
        f"Max # distinct values per attribute    : "
        f"{max_distinct_values_per_attribute:,}"
    )


    print(
        f"Average # distinct values per attribute: "
        f"{average_distinct_values_per_attribute:,.2f}"
    )


    print(
        f"Total distinct values in dataset       : "
        f"{total_distinct_values_dataset:,}"
    )


    print(
        f"Total # clusters                       : "
        f"{number_of_clusters:,}"
    )


    # ========================================================
    # TABLE SUMMARY
    # ========================================================

    print()
    print("=" * 90)
    print("TABLE SUMMARY")
    print("=" * 90)


    for table in table_reports:

        print()

        print(
            f"Table             : "
            f"{table['table']}"
        )

        print(
            f"Rows              : "
            f"{table['rows']:,}"
        )

        print(
            f"Attributes        : "
            f"{table['attributes']:,}"
        )

        print(
            f"Size (MB)         : "
            f"{table['size_mb']:,.2f}"
        )

        print(
            f"Max distinct      : "
            f"{table['max_distinct']:,}"
        )

        print(
            f"Average distinct  : "
            f"{table['average_distinct']:,.2f}"
        )


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {

        "file_type":
            file_type,

        "number_of_tables":
            len(files),

        "dataset_size_mb":
            total_size_mb,

        "total_rows":
            total_rows,

        "max_rows_per_table":
            max_rows_per_table,

        "total_attributes":
            total_attributes,

        "max_distinct_values_per_attribute":
            max_distinct_values_per_attribute,

        "average_distinct_values_per_attribute":
            average_distinct_values_per_attribute,

        "total_distinct_values_dataset":
            total_distinct_values_dataset,

        "number_of_clusters":
            number_of_clusters,

        "tables":
            table_reports
    }


# ============================================================
# RUN
# ============================================================

report = analyze_dataset(
    DATASET_PATH,
    chunk_size=CHUNK_SIZE
)

DATASET ANALYSIS
Dataset path : /Users/gupta/Documents/DIS-IND/data/uniprot
File type    : .tsv
Separator    : '\t'
Tables       : 263

Analyzing: Acanthochromis_polyacanthus.ASM210954v1.111.uniprot.tsv
Detected attributes: 9
Rows               : 34,949
Attributes         : 9
Size (MB)          : 3.40
Max distinct       : 34,949
Average distinct   : 14,168.67

Analyzing: Accipiter_nisus.Accipiter_nisus_ver1.0.111.uniprot.tsv
Detected attributes: 9
Rows               : 2
Attributes         : 9
Size (MB)          : 0.00
Max distinct       : 2
Average distinct   : 1.56

Analyzing: Ailuropoda_melanoleuca.ASM200744v2.111.uniprot.tsv
Detected attributes: 9
Rows               : 18,160
Attributes         : 9
Size (MB)          : 1.68
Max distinct       : 18,095
Average distinct   : 7,607.56

Analyzing: Amphilophus_citrinellus.Midas_v5.111.uniprot.tsv
Detected attributes: 9
Rows               : 31,763
Attributes         : 9
Size (MB)          : 3.09
Max distinct       : 31,763
Average distinct 